# 🔎 Agent 可观测性（Logs、Traces 与 Metrics）

**欢迎来到 Kaggle 5 天 Agents 课程 Day 4！**

在 Day 3，你已经学习了 Session 与 Memory 管理的 **What / Why / How**，重点是长期、短期与共享记忆（state）。

今天你将学习：
- 如何给已构建的 Agent 增加可观测性
- 如何评估 Agent 是否按预期工作

本 Notebook 先聚焦第一部分：**Agent Observability**。

## 什么是 Agent Observability？

**🚨 挑战：** 与传统软件“可预测失败”不同，AI Agent 往往“神秘失败”。例如：

```
User: "Find quantum computing papers"
Agent: "I cannot help with that request."
You: 😭 WHY?? Is it the prompt? Missing tools? API error?
```

**💡 解决方案：** 可观测性让你完整看到 Agent 决策链路：LLM 收到的 prompt、可用工具、模型响应、失败位置都一目了然。

```
DEBUG Log: LLM Request shows "Functions: []" (no tools!)
You: 🎯 Aha! Missing google_search tool - easy fix!
```

## Agent 可观测性的三大支柱

1. **Logs：** 单个事件记录，回答某一时刻**发生了什么**。
2. **Traces：** 把日志串成完整链路，解释最终结果**为什么会发生**。
3. **Metrics：** 汇总指标（平均值、错误率等），反映整体**运行表现如何**。

<center>
    <img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/observability-intro.png">
</center>

**在本 Notebook 中，你将：**

* ✅ 配置 logging
* ✅ 构建一个故意出错的 agent，并用 `adk web` 与 logs 精确定位失败原因
* ✅ 理解如何在生产环境实现 logging
* ✅ 知道何时使用内置 logging，何时自定义

## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖，因此本 Notebook 无需额外安装。

若你要在课程外的本地 Python 环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 🔑 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/)，需要 API key。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，把 API key 作为 Kaggle User Secret 添加到 Notebook。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元，读取你保存的 `GOOGLE_API_KEY` 并设置为环境变量：

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


### ✍️ 1.3：配置日志并清理旧文件
下面先配置日志用于调试，并确保同时采集 DEBUG 等更细粒度日志。

In [3]:
import logging
import os

# Clean up any previous logs
for log_file in ["logger.log", "web.log", "tunnel.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)
        print(f"🧹 Cleaned up {log_file}")

# Configure logging with DEBUG log level.
logging.basicConfig(
    filename="logger.log",
    level=logging.DEBUG,
    format="%(filename)s:%(lineno)s %(levelname)s:%(message)s",
)

print("✅ Logging configured")

✅ Logging configured


### 💻 1.4：配置代理与隧道

我们会在 Kaggle Notebooks 环境中通过代理访问 ADK web UI。如果你在 Kaggle 之外运行，这一步不需要。

In [4]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


---
## 🐞 第 2 部分：用 ADK Web UI 进行实战调试

### 2.1：创建一个“Research Paper Finder” Agent

**目标：** 构建一个可帮助用户查找学术论文的 Agent。

不过我们会先故意做一个“错误版本”，用于练习调试。首先通过 `adk create` CLI 命令创建新的 agent 文件夹。

In [5]:
!adk create research-agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/research-agent:
- .env
- __init__.py
- agent.py



### Agent 定义

接下来创建 root agent。
- 我们把它配置为 `LlmAgent`，并设置 name、model、instruction。
- `root_agent` 接收用户请求，再把搜索任务委派给 `google_search_agent`。
- 然后调用 `count_papers` 工具统计返回论文数量。

**👉 请重点关注 root agent 的 instruction 与 `count_papers` 的参数类型。**

In [6]:
%%writefile research-agent/agent.py

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.agent_tool import AgentTool
from google.adk.tools.google_search_tool import google_search

from google.genai import types
from typing import List

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

# ---- Intentionally pass incorrect datatype - `str` instead of `List[str]` ----
def count_papers(papers: str):
    """
    This function counts the number of papers in a list of strings.
    Args:
      papers: A list of strings, where each string is a research paper.
    Returns:
      The number of papers in the list.
    """
    return len(papers)


# Google Search agent
google_search_agent = LlmAgent(
    name="google_search_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description="Searches for information using Google search",
    instruction="""Use the google_search tool to find information on the given topic. Return the raw search results.
    If the user asks for a list of papers, then give them the list of research papers you found and not the summary.""",
    tools=[google_search]
)


# Root agent
root_agent = LlmAgent(
    name="research_paper_finder_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Your task is to find research papers and count them. 

    You MUST ALWAYS follow these steps:
    1) Find research papers on the user provided topic using the 'google_search_agent'. 
    2) Then, pass the papers to 'count_papers' tool to count the number of papers returned.
    3) Return both the list of research papers and the total number of papers.
    """,
    tools=[AgentTool(agent=google_search_agent), count_papers]
)

Overwriting research-agent/agent.py


### 2.2：运行 Agent

现在用 `adk web --log_level DEBUG` 启动 agent。

**📍 关键在 `--log_level DEBUG`**，它会显示：

* **完整 LLM Prompt：** 发给模型的完整请求（系统指令、历史、工具等）
* 各服务的详细 API 响应
* 内部状态变化与变量值

其他日志级别还包括：INFO、ERROR、WARNING。

获取代理 URL（用于在 Kaggle Notebooks 环境访问 ADK web UI）：

In [7]:
url_prefix = get_adk_proxy_url()

现在可以带上 `--log_level` 参数启动 ADK web UI。

👉 **注意：** 下方单元不会“结束”，它会持续运行并提供 ADK web UI，直到你手动停止。

In [8]:
!adk web --log_level DEBUG --url_prefix {url_prefix}

/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [88]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http:/

ADK web UI 启动后，点击上一单元按钮打开代理链接。

开始与 Agent 对话后，你会在下方输出中看到 DEBUG 日志持续出现。

‼️ **重要：不要把代理链接分享给任何人**，该 URL 中包含鉴权 token，属于敏感信息。

### 📝 2.3：在 ADK web UI 中测试 Agent

#### **👉 操作：在 ADK web UI 中**

1. 左上角下拉选择 "research-agent"
2. 在聊天框输入：`Find latest quantum computing papers`
3. 发送并观察响应。Agent 应返回论文列表及数量统计

看起来 Agent 能回复 🤔 **但等等，论文数量是否异常偏大？我们查看 logs 和 trace。**

#### **👉 操作：Events 标签 - 详细 Trace**

1. 在 web UI 左侧点击 **"Events"**
2. 你会看到按时间排序的 Agent 行为列表
3. 点击任意事件可在底部面板展开详情
4. 可尝试点击 **"Trace"** 查看每一步耗时
5. **点击 `execute_tool count_papers` span，可看到 `count_papers` 返回了异常大的数字**
6. 继续检查传给该函数的输入
7. **找到与 `count_papers` 调用对应的 `call_llm` span**

#### **👉 操作：在 Events 中检查函数调用**

- 点击对应 span 打开 Events 详情
- 查看 `function_call`，重点看 `papers` 参数
- 你会发现 `root_agent` 把 `papers` 作为 **str** 传入，而不是 **List[str]**，这就是 bug

![Demo](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/observability-demo.gif)

### 2.4：轮到你修复！👾

把 `count_papers` 工具中 `papers` 参数的数据类型改为 `List[str]`，然后重新运行 `adk web`。

---

## ‼️ **停止 ADK web UI** 🛑

**为了继续运行本 Notebook 后续单元，** 请先停止你在 3.1 启动 `adk web` 的那个运行中单元。

否则该单元会持续占用，阻塞后续单元执行。

---

### 2.5：通过本地 Logs 调试

你也可以选择直接查看本地 DEBUG 日志定位根因。运行下面单元打印日志文件内容，关注类似：
```
DEBUG - google_adk.models.google_llm - LLM Request: ...
DEBUG - google_adk.models.google_llm - LLM Response: ...
```

In [9]:
# Check the DEBUG logs from the broken agent
print("🔍 Examining web server logs for debugging clues...\n")
!cat logger.log

🔍 Examining web server logs for debugging clues...



**通过 logs 与 adk web，你现在还可以回答这些可观测性问题：**
- **效率**：Agent 的工具选择是否最优？
- **推理质量**：Prompt 结构与上下文是否合理？
- **性能**：通过 trace 看哪些步骤最耗时？
- **故障定位**：出错时具体是哪个环节失败？

**核心收获：** 调试模式是 `symptom -> logs -> root cause -> fix`。

**调试胜利：** 你已经从“Agent 神秘失败”进化到“我知道它为什么失败、如何修复”！这就是可观测性的价值。

---
## 🧑‍💻 第 3 部分：生产环境中的 Logging

**🎯 很好！你已经能用 ADK web UI 与 DEBUG logs 调试 Agent 失败。**

但当你进入生产环境会怎样？以下场景意味着不能只依赖 web UI：

**❌ 场景 1：生产部署**
```
You: "Let me open the ADK web UI to check why the agent failed"
DevOps: "Um... this is a production server. No web UI access."
You: 😱 "How do I debug production issues?"
```

**❌ 场景 2：自动化系统** 
```
You: "The agent runs 1000 times per day in our pipeline"
Boss: "Which runs are slow? What's our success rate?"
You: 😰 "I'd have to manually check the web UI 1000 times..."
```

**💡 解决方案：**

我们需要系统化沉淀可观测数据，也就是**在代码里增加日志**。

👉 在传统软件开发里，这通常通过在 Python 函数中写日志语句实现；**Agent 也完全一样。** 常见做法是在 **Plugins** 中写日志逻辑。

### 3.1：如何为生产可观测性添加日志？

Plugin 是在 Agent 生命周期各阶段自动执行的自定义代码模块。Plugin 由“**Callbacks**”组成，提供打断并挂载逻辑的钩子。可以这样理解：

- **Agent 工作流**：用户消息 -> Agent 思考 -> 调用工具 -> 返回响应
- **Plugin 插入点**：Agent 开始前 -> 工具执行后 -> LLM 返回时 -> 等等
- **Plugin 里的自定义逻辑**：日志、监控、安全检查、缓存等

![image.png](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/plugins-callbacks.png)

#### Callbacks

Callbacks 是 **Plugin 内部的原子组件**：本质上就是在 Agent 生命周期特定节点执行的 Python 函数。**多个 callbacks 组合成一个 Plugin。**

常见 callback 类型包括：
* **before/after_agent_callbacks** - Agent 调用前后
* **before/after_tool_callbacks** - 工具调用前后
* **before/after_model_callbacks** - LLM 调用前后
* **on_model_error_callback** - 发生模型错误时

![image.png](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/types_of_callbacks.png)

### 3.2：具体来看，Plugin 长什么样？

In [10]:
print("----- EXAMPLE PLUGIN - DOES NOTHING ----- ")

import logging
from google.adk.agents.base_agent import BaseAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.plugins.base_plugin import BasePlugin


# Applies to all agent and model calls
class CountInvocationPlugin(BasePlugin):
    """A custom plugin that counts agent and tool invocations."""

    def __init__(self) -> None:
        """Initialize the plugin with counters."""
        super().__init__(name="count_invocation")
        self.agent_count: int = 0
        self.tool_count: int = 0
        self.llm_request_count: int = 0

    # Callback 1: Runs before an agent is called. You can add any custom logic here.
    async def before_agent_callback(
        self, *, agent: BaseAgent, callback_context: CallbackContext
    ) -> None:
        """Count agent runs."""
        self.agent_count += 1
        logging.info(f"[Plugin] Agent run count: {self.agent_count}")

    # Callback 2: Runs before a model is called. You can add any custom logic here.
    async def before_model_callback(
        self, *, callback_context: CallbackContext, llm_request: LlmRequest
    ) -> None:
        """Count LLM requests."""
        self.llm_request_count += 1
        logging.info(f"[Plugin] LLM request count: {self.llm_request_count}")

----- EXAMPLE PLUGIN - DOES NOTHING ----- 


**关键洞察**：你只需在 runner 上**注册一次** plugin，它就会按定义自动作用于系统中的**每个 agent、每次工具调用和每次 LLM 请求**。更多 Plugin hooks 说明见[这里](https://google.github.io/adk-docs/plugins/#plugin-callback-hooks)。

你可以对照下图中的编号来理解整个执行流。

![image.png](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/count-invocation-plugin.png)

### 3.3：ADK 内置 `LoggingPlugin`

如果你只需要采集 ADK 标准可观测数据，不必手写全部 callbacks 与 plugins。ADK 提供内置 **LoggingPlugin**，可自动捕获：

- 🚀 用户消息与 Agent 响应
- ⏱️ 性能分析所需时序数据
- 🧠 用于调试的 LLM 请求与响应
- 🔧 工具调用及结果
- ✅ 完整执行 traces

#### Agent 定义

继续沿用上一节 demo 的 Research paper finder agent。

In [11]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.agent_tool import AgentTool
from google.adk.tools.google_search_tool import google_search

from google.genai import types
from typing import List

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)


def count_papers(papers: List[str]):
    """
    This function counts the number of papers in a list of strings.
    Args:
      papers: A list of strings, where each string is a research paper.
    Returns:
      The number of papers in the list.
    """
    return len(papers)


# Google search agent
google_search_agent = LlmAgent(
    name="google_search_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description="Searches for information using Google search",
    instruction="Use the google_search tool to find information on the given topic. Return the raw search results.",
    tools=[google_search],
)

# Root agent
research_agent_with_plugin = LlmAgent(
    name="research_paper_finder_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Your task is to find research papers and count them. 
   
   You must follow these steps:
   1) Find research papers on the user provided topic using the 'google_search_agent'. 
   2) Then, pass the papers to 'count_papers' tool to count the number of papers returned.
   3) Return both the list of research papers and the total number of papers.
   """,
    tools=[AgentTool(agent=google_search_agent), count_papers],
)

print("✅ Agent created")

✅ Agent created


### 3.4：把 LoggingPlugin 加到 Runner

下方代码创建 `InMemoryRunner`，用于以编程方式调用 Agent。

**要在该 research agent 中使用 `LoggingPlugin`：**
1) 导入 plugin
2) 在初始化 `InMemoryRunner` 时加入

In [12]:
from google.adk.runners import InMemoryRunner
from google.adk.plugins.logging_plugin import (
    LoggingPlugin,
)  # <---- 1. Import the Plugin
from google.genai import types
import asyncio

runner = InMemoryRunner(
    agent=research_agent_with_plugin,
    plugins=[
        LoggingPlugin()
    ],  # <---- 2. Add the plugin. Handles standard Observability logging across ALL agents
)

print("✅ Runner configured")

✅ Runner configured


现在用 `run_debug` 运行该 Agent。

In [ ]:
print("🚀 Running agent with LoggingPlugin...")
print("📊 Watch the comprehensive logging output below:\n")

response = await runner.run_debug("Find recent papers on quantum computing")

🚀 Running agent with LoggingPlugin...
📊 Watch the comprehensive logging output below:


 ### Created new session: debug_session_id

User > Find recent papers on quantum computing
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-f650796f-db9c-489e-845d-7f9d67b86bff
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: research_paper_finder_agent
[logging_plugin]    User Content: text: 'Find recent papers on quantum computing'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-f650796f-db9c-489e-845d-7f9d67b86bff
[logging_plugin]    Starting Agent: research_paper_finder_agent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: research_paper_finder_agent
[logging_plugin]    Invocation ID: e-f650796f-db9c-489e-845d-7f9d67b86bff
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: gemini-2.5-flash-lite
[l

---

## 📊 总结

**❓该用哪种 Logging？**
1. **开发调试阶段？** -> 用 `adk web --log_level DEBUG`
2. **通用生产可观测？** -> 用 `LoggingPlugin()`
3. **有定制化需求？** -> 自定义 Callbacks 与 Plugins

### 动手试试

👉 进一步扩展可观测性：实现一个**自定义 ADK plugin**，统计并上报单次 session 内工具调用总次数。

## 🎯 恭喜！

**你现在已经会：**

- ✅ 通过 DEBUG logs 与 ADK web UI 调试 Agent 故障
- ✅ 使用核心调试模式：symptom -> logs -> root cause -> fix  
- ✅ 在生产系统中用 `LoggingPlugin` 规模化可观测性
- ✅ 根据场景选择不同日志方案

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 资源

**更多可观测性内容可参考 ADK 文档：**

- [ADK Observability Documentation](https://google.github.io/adk-docs/observability/logging/) - ADK 日志完整指南
- [Custom Plugin](https://google.github.io/adk-docs/plugins/) - 自定义 Plugins
- [External Integrations](https://google.github.io/adk-docs/observability/cloud-trace/) - ADK 与第三方可观测系统集成

### 🎯 下一步

准备好下一挑战了吗？继续下一本 Notebook，学习如何**评估 Agent（Evaluate an Agent）**，确保其在生产环境稳定达标。